# Crypto Portfolio Analytics

Discovers live perp positions and spot balances in configured Hyperliquid wallets, uses Hyperliquid daily candles for perp markets and CoinGecko prices and market data for supported assets, then calculates correlations and constrained long/short full-Kelly weights and reports portfolio performance, risk, and factor estimates. Implementation lives in `crypto_portfolio_analytics_implementation.ipynb`.


## Load implementation

Set the wallet addresses to analyze, then load the companion implementation and create the analytics object.


In [1]:
import json
from pathlib import Path

WALLET_ADDRESSES = [
    '0xf3fb3598bFfA608d1e7CF2837d74F02B1191c248',
    '0x7532dA34edA8f3e383bC480d50e7A7d3b12c07f2',
]


implementation_path=Path('crypto_portfolio_analytics_implementation.ipynb')
implementation=json.loads(implementation_path.read_text(encoding='utf-8'))
namespace={'__name__':'crypto_portfolio_analytics'}
for cell in implementation['cells']:
    if cell['cell_type']=='code':
        source=''.join(cell['source'])
        if source.strip():
            exec(compile(source,f'{implementation_path} cell','exec'),namespace)
CryptoPortfolioAnalytics=namespace['CryptoPortfolioAnalytics']
analysis=CryptoPortfolioAnalytics(data_dir='data', wallet_addresses=WALLET_ADDRESSES)


## Market data refresh

Discovers current nonzero perp positions and spot balances. Perp prices come from Hyperliquid daily candles (including HIP-3 markets); CoinGecko supplies spot-only prices and market-cap/volume data without requiring an API key. Assets without enough common daily returns against BTC are reported and skipped. Edit `WALLET_ADDRESSES` above to select wallets.


In [2]:
analysis.refresh_market_data()


RuntimeError: BTC: CoinGecko public API request failed (ReadTimeout).

## Price history

Loads assets discovered from wallet positions and spot balances, plus BTC for benchmark comparisons.


In [ ]:
analysis.load_price_history()


## Daily returns and correlations

Calculates the complete daily-return sample and Pearson correlations.


In [ ]:
analysis.calculate_returns()


## Correlation heatmap

Displays the daily return correlation matrix as an interactive Plotly heatmap.


In [ ]:
analysis.plot_correlations()


## Constrained long/short full-Kelly portfolio

Optimizes signed asset weights from historical returns; the remaining net balance is held as cash. Long/short signs are chosen by the optimizer and are not forced to match current wallet position sides.


In [ ]:
analysis.optimize_portfolio()


Constrained long/short full-Kelly weights, sorted by coin weight (%)
Gross long exposure: 150.0%; gross short exposure: 180.2%; net exposure: -30.2%.
Expected price PnL is a historical mean-forward-return contribution per $100 of portfolio equity; it excludes funding, fees, and slippage.


,Weight (%),Expected Price PnL / $100 equity (5d)
AVAX,15.00,$-0.087
GRASS,15.00,$0.345
INIT,15.00,$0.217
AERO,15.00,$0.041
RSR,15.00,$-0.178
ZRO,15.00,$0.135
BTC,15.00,$-0.046
xyz:NET,15.00,$0.033
HYPE,15.00,$0.392
xyz:AAPL,15.00,$0.058


## Portfolio performance and risk metrics

Displays the Plotly portfolio-vs-BTC chart and performance and risk tables. Return and volatility annualization use 365 calendar days. Arithmetic annualized return is the daily mean multiplied by 365, not a forecast or compounded return. Historical VaR is an empirical quantile; Normal VaR is a Gaussian-model comparison. The observation and tail counts show how limited the 99% historical estimate is.


In [ ]:
analysis.analyze_performance()


,Return
1D,-1.45%
7D,15.09%
M (1 month),47.21%
Q (3 months),N/A
Since Inception,148.85%


,Daily return
Average daily return,1.73%
Largest daily gain,9.93%
Largest daily loss,-3.21%


,All-Time High ($),Minimum Portfolio Value ($),Max Drawdown ($),Max Drawdown (%)
Since inception,$259.70,$100.00,-$14.27,-6.69%


,Sharpe,Beta,Alpha,Expected Return,Hist Vol (daily),Hist Vol (annualized),1D Hist VaR 95%,VaR 95% ($ / $100),1D Hist VaR 99%,VaR 99% ($ / $100),Risk-Free Rate,Max Drawdown
Constrained long/short full-Kelly,9.661,-0.617,625.15%,630.05%,3.39%,64.71%,2.90%,$2.90,3.17%,$3.17,5.00%,-6.69%


## CoinGecko factor model

Fits factor exposures and displays the annual factor estimates and model fit.


In [ ]:
analysis.fit_factor_model()


Factor model: R²=10.8% | adjusted R²=9.1% | annual alpha=625.15% | n=55 days
Single-factor BTC beta=-0.617 | factor-model BTC beta=-0.617
CoinGecko cross-sectional factors exclude assets without market-cap history: AVAX, GRASS, TRUMP, INIT, LIT, USELESS, TRX, ADA, xyz:GOLD, xyz:AAPL, xyz:GOOGL, xyz:NET, xyz:RDDT. Cross-sectional factor model had only 22 complete observations; using the BTC-only model to retain the available portfolio history.


,Beta,HAC t-stat,Factor premium (%/yr),Portfolio contribution (%/yr)
BTC market excess,-0.617,-4.61,-0.03%,0.02%
Alpha,N/A,3.93,N/A,625.15%


,Factor model fit
Observations,55.000000
R-squared,0.107695
Adjusted R-squared,0.090859
Annual alpha,6.251538
BTC beta (factor model),-0.617397
BTC beta (single factor),-0.617397


## Save analysis outputs

Writes calculated tables and series to CSV files in `data/`.


In [ ]:
analysis.save_outputs()


Saved portfolio, factor, market, and return outputs to data/.
